# Interactive charts and dashboards

*Plotly for exploration, Streamlit for the dashboard*

In [Chapter 4](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-04-single-variable.html) and [Chapter 5](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-05-relationships.html) we made charts to answer questions we had already asked. Charts have two further uses in an analyst's work. The first use is exploration: we make many quick charts to discover patterns we may not have thought about, or questions we may want to explore further. The second use is communication: we put live, filterable answers in front of a manager who does not read code. In this chapter we do both in Python. We explore with Plotly's interactive charts. With **Streamlit**, we then turn our pandas and Plotly work into a dashboard application in about forty lines. We close Part II here, at the communication stage of the course map.

> **Setup for this chapter**
>
> You can run this chapter's notebook in two ways. In the cloud, [**open ch-06-dashboards.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-06-dashboards.ipynb); nothing needs to be installed. Locally, use the `pyba-core` environment (Appendix A). Either way, run the setup cell below first. The dashboard in the second half runs in both places: locally with `streamlit run`, and on Colab with the launcher cell provided in that section. No keys or paid accounts are needed anywhere in this book; Colab requires only a free Google account.

In [ ]:
# Setup. Run this cell once per session. It installs this chapter's
# packages; on Google Colab it also fetches the course data.
%pip install -q pandas plotly streamlit
import sys
if "google.colab" in sys.modules:
    !test -d pyba-companion || git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

> **Tools in this chapter**
>
> | Tool | Why we use it here | Alternatives | Trade-off |
> |---|---|---|---|
> | Streamlit | Turns a Python script into a web dashboard; reruns the script when a control changes, so there is no separate web framework to learn | Dash (more control, explicit callbacks), Panel (more flexible, steeper), BI tools such as Tableau and Power BI | Streamlit is the fastest path from analysis to dashboard, and the skill remains useful in more advanced analytics work |
>
## Exploration with Plotly
Data exploration is an important part of analytics. It is practical only when the analyst can slice the data and adjust the visualizations quickly. Our Plotly charts already have three behaviors of this kind, with no extra code: we can see exact values by hovering, zoom by dragging, and hide or isolate a group through its legend entry (one click hides the group, a double-click shows it alone). Beyond these behaviors, most early questions about a dataset can be addressed with a few standard combinations of a grouping, an aggregation, and a chart type. We start with the joined line-level table from [Chapter 3](https://pyba.murtaza.cc/parts/part-01-foundations/ch-03-pandas.html) and go through three such combinations.

In [ ]:
import pandas as pd
import plotly.express as px

from pyba import DATA_DIR

orders = pd.read_csv(DATA_DIR / "pw_orders.csv", parse_dates=["order_date"])
products = pd.read_csv(DATA_DIR / "pw_products.csv")
customers = pd.read_csv(DATA_DIR / "pw_customers.csv")

product_cols = products[["sku", "product_name", "category", "unit_cost"]]
lines = orders.merge(product_cols, on="sku", how="left", validate="many_to_one")
lines = lines.merge(customers[["customer_id", "region", "business_type"]],
                    on="customer_id", how="left", validate="many_to_one")
lines["margin_dollars"] = lines["line_total"] - lines["quantity"] * lines["unit_cost"]
len(lines)

### Revenue over time at a chosen grain

The **grain** is the level of time aggregation: daily, weekly, or monthly. We set the grain with `resample`, which allows us to ask the question at different timescales: at the monthly grain we see the seasonality, and at the weekly grain we mostly see variation due to noise, since our dataset does not exhibit weekly patterns. Depending on the dataset you are working with, patterns may appear at different grains or timescales.

In [ ]:
monthly = lines.set_index("order_date")["line_total"].resample("ME").sum().reset_index()
px.line(monthly, x="order_date", y="line_total",
        labels={"order_date": "", "line_total": "Monthly revenue ($)"})

::: {.content-visible when-format="html"}
The explorer below shows the same revenue series at all three grains. Switch between them with the buttons and compare what you can see at each grain.

<iframe src="../../assets/demos/time-grain.html" width="100%" height="470" style="border:1px solid #d0d7de; border-radius:8px;" title="Time grain explorer"></iframe>
:::

::: {.content-visible unless-format="html"}
The online edition has a grain explorer here: the same revenue series, switchable between the daily, weekly, and monthly grains. @fig-monthly-revenue shows the monthly grain.
:::

### A grouped comparison

The next chart shows revenue by category, split by region and sorted. We produce the comparison with one call:

In [ ]:
by_cat = (lines.groupby(["region", "category"], as_index=False)["line_total"].sum()
          .sort_values("line_total", ascending=False))
px.bar(by_cat, x="line_total", y="category", color="region", orientation="h",
       labels={"line_total": "Revenue ($)", "category": ""})

### A summary by group

In the third combination, we chart one summary number per group. Before creating the chart, we must choose the summary: in [Chapter 4](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-04-single-variable.html) we saw that the mean and the median of order totals differ by a factor of almost three, and that the median is the right summary for the typical order in each channel. The choice also belongs on the chart itself: a viewer cannot tell from a bar whether it shows a mean or a median, so we label every visualization with the aggregation behind its numbers. Here the label is on the axis:

In [ ]:
order_totals = lines.groupby("order_id").agg(total=("line_total", "sum"),
                                             channel=("channel", "first"))
med = order_totals.groupby("channel", as_index=False)["total"].median()
px.bar(med, x="channel", y="total",
       labels={"channel": "", "total": "Median order total ($)"})

In a few cells we answered three questions. Exploration consists of repeating this loop: ask, group, plot, look, ask again. We practiced every part of the loop in [Chapter 3](https://pyba.murtaza.cc/parts/part-01-foundations/ch-03-pandas.html) and [Chapter 4](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-04-single-variable.html). In the remainder of this chapter, we take the next step: we package the loop's results for decision makers, who will consume the analysis but may never open a Python notebook.

## What a dashboard adds

A notebook is written for its author. A **dashboard** is built for an audience: it presents a fixed set of charts and numbers, it updates when the data updates, and a non-programmer can filter it to the part of the business they manage. When the regional manager selects "Southwest", every number recomputes without a request to an analyst and without any code being edited.

**Streamlit** is an open-source Python library built for exactly this purpose. A Streamlit app is a plain Python script that runs top to bottom. Everything the script declares (for example, a metric, a chart, or a filter widget) is shown on a web page. When the user moves a control, Streamlit reruns the script with the new value. There are no callbacks to write and no web technology to learn. As such, a working dashboard is about forty lines of code.

## The Monday-morning dashboard

The companion repository includes the full app as `dashboards/monday_dashboard.py`. Run it from the repository root:

```bash
streamlit run dashboards/monday_dashboard.py
```

A browser tab opens with the dashboard: a row of key performance indicators (KPIs), a revenue trend, a category comparison, and two sidebar filters. When a filter changes, everything recomputes.

On **Colab** there is no terminal, so the cell below launches the dashboard. It starts the Streamlit server in your Colab session and, through Colab's port proxy, opens the dashboard in a new browser tab. Explore the dashboard in that tab. To stop the app, interrupt the notebook cell or restart the runtime.

In [ ]:
# Colab only: launch the dashboard and open it in a new tab.
import os, subprocess, sys, time

if "google.colab" in sys.modules:
    env = {**os.environ, "PYTHONPATH": "pyba-companion"}   # so the app finds pyba
    subprocess.Popen(
        ["streamlit", "run", "pyba-companion/dashboards/monday_dashboard.py",
         "--server.headless=true", "--server.port=8501",
         "--server.enableCORS=false", "--server.enableXsrfProtection=false"],
        env=env)
    time.sleep(8)                                          # give the server a moment
    from google.colab import output
    output.serve_kernel_port_as_window(8501)               # prints a link; click it

The script has four parts. Each part uses ideas from earlier chapters.

### Cached loading

We load the data with the same pandas calls as in every chapter. With the `@st.cache_data` line, Streamlit loads the data once and reuses the result between reruns, so the 140,000 rows are not reread every time a slider moves:

```python
import streamlit as st

@st.cache_data
def load_lines():
    orders = pd.read_csv(DATA_DIR / "pw_orders.csv", parse_dates=["order_date"])
    ...                       # the Chapter 3 joins, unchanged
    return lines
```

### Filters

The audience's view is set with two sidebar controls. The controls' current values are regular Python variables. The filter itself is the [Chapter 3](https://pyba.murtaza.cc/parts/part-01-foundations/ch-03-pandas.html) boolean mask:

```python
regions = st.sidebar.multiselect("Region", sorted(lines["region"].unique()))
start, end = st.sidebar.slider("Date range", ...)

view = lines[(lines["order_date"].dt.date >= start)
             & (lines["order_date"].dt.date <= end)]
if regions:
    view = view[view["region"].isin(regions)]
```

### The KPI row

The row holds three headline numbers. The third is unique orders, computed with `nunique`: the table is line-level, so a plain row count would count order lines, and an order spans several lines. This is the [Chapter 3](https://pyba.murtaza.cc/parts/part-01-foundations/ch-03-pandas.html) lesson about keeping in mind what each table row represents:

```python
k1, k2, k3 = st.columns(3)
k1.metric("Revenue", f"${view['line_total'].sum():,.0f}")
k2.metric("Margin", f"${view['margin_dollars'].sum():,.0f}")
k3.metric("Orders", f"{view['order_id'].nunique():,}")
```

### The charts

The charts are the same Plotly figures from the exploration section, passed to `st.plotly_chart`. The figures are built from `view`, so they are recalculated on every filter change.

Nothing in these four parts is new analysis. The loading, the filters, and the charts are the pandas and Plotly work of earlier chapters; the new code is the handful of Streamlit calls that place them on a web page. Only the organization changed: the same analysis is now arranged around an audience and a filter.

> **Side note: the BI tools you will meet at work**
>
> Many companies host dashboards in commercial BI tools such as Tableau or Power BI. Their concepts map directly onto what you just learned: their "dimensions" are groupby keys, their "measures" are columns with aggregations, and their aggregation menus present the same median-versus-mean choice we made above. An analyst who knows the pandas version can learn any of these tools quickly. The Python version you just built has three advantages: one language end to end, no license cost, and version control shared with the analysis code.

## Design rules for dashboards

The literature on dashboard design is large. It can be compressed into three rules that apply in any tool.

- **One question per dashboard.** The Monday-morning dashboard is built around one question: how has the business been doing recently, across regions and categories? Despite the filters and the multiple charts, that is one question. If the churn analysis were added to the same screen, the result would be two half dashboards.
- **The ten-second test.** A first-time viewer should be able to identify the headline (what is big, what is up, what is down) in ten seconds. If they have to study a legend to orient themselves, simplify.
- **Show the aggregation.** Label whether a number is a sum, a median, or a distinct count. Without the label, the viewer cannot tell what was computed.

## Sharing the dashboard

`streamlit run` hosts the app on your machine. Anyone on the same network can open it at the address shown by Streamlit. For permanent deployment, the app can be set up on a small server in the same way. Streamlit's hosted service (Community Cloud) can host an app from a GitHub repository. The service requires an account, so we do not use it in this course. The reproducibility standard from [Chapter 1](https://pyba.murtaza.cc/parts/part-01-foundations/ch-01-analytics-and-python.html) now applies to the dashboard: when the November data is available, the same script shows the new numbers. Every number and chart on the dashboard comes from code stored in the repository, so anyone who questions a number can trace how it was produced.

## Evaluation: is the dashboard effective?

The concrete test for your build-lab dashboard is the ten-second test, run on another person: show them the dashboard silently for ten seconds, then ask for the headline. If their answer matches the headline you intended, the dashboard passes the test. If they describe the layout instead (for example, "there are some bars and a line"), the dashboard fails the test; in that case, reduce the number of visual elements and clarify the sorting. In a few minutes, we measure the outcome itself: whether the intended headline was received.

## Exercises



### Build lab

Extend `monday_dashboard.py`. Add a channel filter to the sidebar (a `st.multiselect` over the three channels, applied to `view` like the region filter). Add a fourth KPI: median order total for the current view, labeled as a median. Add one new chart of your choosing that answers a stated question, with the aggregation named in its axis label. Run the app locally with `streamlit run`, or in Colab with the launcher notebook named in the callout below, and submit the modified script plus a screenshot.

### Evaluate lab

Run the ten-second test on another person with your extended dashboard. Record what they said the headline was and whether it matched your intent. If it did not match, make one change in response, rerun the test, and report whether the answer improved. Submit the answers and the change, or the single passing answer, in at most three sentences.

> **Lab starter**
>
> A starter script for this lab is provided as `lab_6_dashboard_starter.py`, a copy of `monday_dashboard.py` with the lab's tasks marked in the places where the code belongs. [**View lab_6_dashboard_starter.py**](https://github.com/murtaza-nasir/pyba-companion/blob/main/dashboards/lab_6_dashboard_starter.py), or download it from the course page in Blackboard.
>
> Working locally, edit that script and run it with `streamlit run`. On Colab, [**open ch-06-lab-starter.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-06-lab-starter.ipynb): it fetches the starter script, opens it for editing in the Colab file browser, and launches it with the same port-proxy launcher used earlier in this chapter. That notebook opens read-only from GitHub, so click **Copy to Drive** in the toolbar before you start.